# 07_03F — Graph Node2Vec Fast Features — Early-Stage

## Objetivo

Testar **embeddings estruturais de grafo com Node2Vec acelerado** para o target:

> `0 = banco_ganhou | 1 = banco_perdeu`

Este notebook testa o sinal estrutural do grafo de forma controlada, antes de misturar com target encoding, texto e calibração.

Ordem de tentativa para embeddings:

1. `pecanpy`, se disponível.
2. `nodevectors`, se disponível.
3. fallback com `TruncatedSVD` na matriz processo–entidade.

> Observação: se o notebook cair no fallback SVD, ele ainda roda, mas a hipótese principal de Node2Vec acelerado não foi testada de fato.

In [ ]:
import os
import re
import json
import time
import pickle
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 300)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 260)

from scipy import sparse

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_recall_curve,
    classification_report,
    confusion_matrix,
)
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

try:
    import lightgbm as lgb
    HAS_LIGHTGBM = True
except Exception as e:
    HAS_LIGHTGBM = False
    print("LightGBM indisponível:", repr(e))

try:
    import xgboost as xgb
    HAS_XGBOOST = True
except Exception as e:
    HAS_XGBOOST = False
    print("XGBoost indisponível:", repr(e))

try:
    import networkx as nx
    HAS_NETWORKX = True
except Exception as e:
    HAS_NETWORKX = False
    print("NetworkX indisponível:", repr(e))

try:
    import pecanpy
    HAS_PECANPY = True
except Exception as e:
    HAS_PECANPY = False
    print("PecanPy indisponível:", repr(e))

try:
    from nodevectors import Node2Vec as NodevectorsNode2Vec
    HAS_NODEVECTORS = True
except Exception as e:
    HAS_NODEVECTORS = False
    print("nodevectors indisponível:", repr(e))

In [ ]:
# ============================================================
# Configurações gerais
# ============================================================

BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
REPORTS_DIR = BASE_DIR / "outputs" / "reports" / "modelagem_early_stage"
MODELS_DIR = BASE_DIR / "outputs" / "models" / "modelagem_early_stage"
ARTIFACTS_DIR = BASE_DIR / "outputs" / "artifacts" / "modelagem_early_stage"

for p in [PROCESSED_DIR, REPORTS_DIR, MODELS_DIR, ARTIFACTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

RUN_MODE = "fast"  # "fast" ou "full"
RANDOM_STATE = 42
TARGET_COL = "target_banco_perdeu"

DEV_FILE = PROCESSED_DIR / "abt_early_stage_v1_dev.parquet"
HOLDOUT_FILE = PROCESSED_DIR / "abt_early_stage_v1_holdout.parquet"
FEATURE_LIST_FILE = PROCESSED_DIR / "feature_list_early_stage_v1.txt"

GRAPH_EDGES_FILE = PROCESSED_DIR / "graph_edges_early_stage_07_03f.csv"
GRAPH_EMB_DEV_FILE = PROCESSED_DIR / "graph_node2vec_fast_embeddings_dev_07_03f.parquet"
GRAPH_EMB_HOLDOUT_FILE = PROCESSED_DIR / "graph_node2vec_fast_embeddings_holdout_07_03f.parquet"
RANKING_RISCO_HOLDOUT_FILE = PROCESSED_DIR / "ranking_holdout_risco_perda_07_03f.parquet"
RANKING_FIN_HOLDOUT_FILE = PROCESSED_DIR / "ranking_holdout_prioridade_financeira_07_03f.parquet"

N2V_DIM = 32 if RUN_MODE == "fast" else 64
N2V_WALK_LENGTH = 16 if RUN_MODE == "fast" else 32
N2V_NUM_WALKS = 4 if RUN_MODE == "fast" else 10
N2V_WINDOW = 5
N2V_EPOCHS = 5 if RUN_MODE == "fast" else 10
N2V_P = 1.0
N2V_Q = 1.0

MIN_ENTITY_FREQ = 20 if RUN_MODE == "fast" else 5
MAX_CATEGORIES_PER_COL = 5000 if RUN_MODE == "fast" else 20000
N_SPLITS = 3 if RUN_MODE == "fast" else 5
TOP_KS = (0.01, 0.05, 0.10, 0.20)

print("RUN_MODE:", RUN_MODE)
print("HAS_PECANPY:", HAS_PECANPY)
print("HAS_NODEVECTORS:", HAS_NODEVECTORS)
print("HAS_LIGHTGBM:", HAS_LIGHTGBM)
print("HAS_XGBOOST:", HAS_XGBOOST)

In [ ]:
# ============================================================
# Funções utilitárias
# ============================================================

def save_report(df, filename, index=False):
    path = REPORTS_DIR / filename
    df.to_csv(path, index=index, encoding="utf-8-sig")
    print(f"Relatório salvo em: {path}")
    return path

def save_parquet(df, path):
    df.to_parquet(path, index=False)
    print(f"Arquivo salvo em: {path}")
    return path

def read_feature_list(path):
    if not Path(path).exists():
        return None
    with open(path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]

def normalize_token(x):
    if pd.isna(x):
        return None
    s = str(x).strip().lower()
    s = re.sub(r"\s+", "_", s)
    s = re.sub(r"[^a-zA-Z0-9_áàâãéèêíïóôõöúçñÁÀÂÃÉÈÊÍÏÓÔÕÖÚÇÑ-]", "", s)
    if s in ["", "nan", "none", "null"]:
        return None
    return s

def get_id_col(df):
    candidates = [
        "id_processo", "processo_id", "process_id", "numero_processo",
        "Numerodistribuicao", "numero_processo_norm", "processo"
    ]
    for c in candidates:
        if c in df.columns:
            return c
    return None

def get_date_col(df):
    candidates = [
        "data_inicio_processo", "data_distribuicao", "DataDistribuicao",
        "data_entrada", "dt_inicio", "ano_inicio_processo", "data_ref"
    ]
    for c in candidates:
        if c in df.columns:
            return c
    return None

def get_value_col(df):
    candidates = [
        "fe_valor_ajuizado", "Valorajuizado", "valor_ajuizado",
        "valor_da_causa", "valor_causa", "Valor_Ajuizado"
    ]
    for c in candidates:
        if c in df.columns:
            return c
    return None

def safe_numeric_series(s):
    return pd.to_numeric(s, errors="coerce").replace([np.inf, -np.inf], np.nan)

def format_elapsed(sec):
    sec = float(sec)
    if sec < 60:
        return f"{sec:.1f}s"
    if sec < 3600:
        return f"{sec/60:.1f}min"
    return f"{sec/3600:.1f}h"

In [ ]:
# ============================================================
# Métricas
# ============================================================

def best_f1_threshold(y_true, proba):
    precision, recall, thresholds = precision_recall_curve(y_true, proba)
    f1 = 2 * precision * recall / np.maximum(precision + recall, 1e-12)
    if len(thresholds) == 0:
        return 0.5, float(np.nanmax(precision)), float(np.nanmax(recall)), float(np.nanmax(f1))
    idx = int(np.nanargmax(f1[:-1]))
    return float(thresholds[idx]), float(precision[idx]), float(recall[idx]), float(f1[idx])

def threshold_metrics(y_true, proba, threshold):
    pred = (proba >= threshold).astype(int)
    tp = int(((pred == 1) & (y_true == 1)).sum())
    fp = int(((pred == 1) & (y_true == 0)).sum())
    fn = int(((pred == 0) & (y_true == 1)).sum())
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)
    return {
        "threshold": threshold,
        "precision_perda": precision,
        "recall_perda": recall,
        "f1_perda": f1,
        "pred_perda_rate": float(pred.mean()),
    }

def topk_risco_perda_metrics(y_true, proba_perda, ks=TOP_KS):
    df = pd.DataFrame({"y_true": np.asarray(y_true), "proba_perda": np.asarray(proba_perda)})
    df = df.sort_values("proba_perda", ascending=False).reset_index(drop=True)
    taxa_base = df["y_true"].mean()
    rows = []
    n = len(df)
    for k in ks:
        n_top = max(1, int(np.ceil(n * k)))
        top = df.head(n_top)
        precision_k = top["y_true"].mean()
        recall_k = top["y_true"].sum() / max(df["y_true"].sum(), 1)
        lift_k = precision_k / max(taxa_base, 1e-12)
        rows.append({
            "top_k": k,
            "n_top": n_top,
            "precision_perda_at_k": precision_k,
            "recall_perda_at_k": recall_k,
            "lift_perda_at_k": lift_k,
            "taxa_perda_base": taxa_base,
        })
    return pd.DataFrame(rows)

def topk_prioridade_financeira_metrics(y_true, proba_perda, valor_ajuizado, ks=TOP_KS):
    df = pd.DataFrame({
        "y_true": np.asarray(y_true),
        "proba_perda": np.asarray(proba_perda),
        "valor_ajuizado": safe_numeric_series(pd.Series(valor_ajuizado)).fillna(0).clip(lower=0).values,
    })
    df["score_financeiro"] = df["proba_perda"] * df["valor_ajuizado"]
    df["valor_ajuizado_perdas"] = df["valor_ajuizado"] * df["y_true"]

    total_valor = df["valor_ajuizado"].sum()
    total_valor_perdas = df["valor_ajuizado_perdas"].sum()
    taxa_base = df["y_true"].mean()

    df = df.sort_values("score_financeiro", ascending=False).reset_index(drop=True)

    rows = []
    n = len(df)
    for k in ks:
        n_top = max(1, int(np.ceil(n * k)))
        top = df.head(n_top)
        precision_k = top["y_true"].mean()
        recall_k = top["y_true"].sum() / max(df["y_true"].sum(), 1)
        lift_k = precision_k / max(taxa_base, 1e-12)
        valor_top = top["valor_ajuizado"].sum()
        valor_perdas_top = top["valor_ajuizado_perdas"].sum()
        rows.append({
            "top_k": k,
            "n_top": n_top,
            "precision_perda_at_k": precision_k,
            "recall_perda_at_k": recall_k,
            "lift_perda_at_k": lift_k,
            "taxa_perda_base": taxa_base,
            "valor_ajuizado_top": valor_top,
            "share_valor_ajuizado_total": valor_top / max(total_valor, 1e-12),
            "valor_ajuizado_perdas_top": valor_perdas_top,
            "share_valor_perdas_total": valor_perdas_top / max(total_valor_perdas, 1e-12),
        })
    return pd.DataFrame(rows)

def evaluate_probs(y_true, proba):
    y_true = np.asarray(y_true)
    proba = np.asarray(proba)
    roc = roc_auc_score(y_true, proba)
    pr = average_precision_score(y_true, proba)
    best_thr, best_p, best_r, best_f1 = best_f1_threshold(y_true, proba)
    thr05 = threshold_metrics(y_true, proba, 0.5)
    return {
        "roc_auc_perda": roc,
        "pr_auc_perda": pr,
        "best_f1_threshold_perda": best_thr,
        "best_f1_precision_perda": best_p,
        "best_f1_recall_perda": best_r,
        "best_f1_perda": best_f1,
        "precision_perda_t05": thr05["precision_perda"],
        "recall_perda_t05": thr05["recall_perda"],
        "f1_perda_t05": thr05["f1_perda"],
        "pred_perda_rate_t05": thr05["pred_perda_rate"],
    }

In [ ]:
# ============================================================
# Carregamento dos dados
# ============================================================

assert DEV_FILE.exists(), f"Arquivo não encontrado: {DEV_FILE}"
assert HOLDOUT_FILE.exists(), f"Arquivo não encontrado: {HOLDOUT_FILE}"

df_dev = pd.read_parquet(DEV_FILE)
df_holdout = pd.read_parquet(HOLDOUT_FILE)

print("df_dev:", df_dev.shape)
print("df_holdout:", df_holdout.shape)

assert TARGET_COL in df_dev.columns, f"{TARGET_COL} não encontrado no dev"
assert TARGET_COL in df_holdout.columns, f"{TARGET_COL} não encontrado no holdout"

raw_id_col = get_id_col(df_dev)
date_col = get_date_col(df_dev)
value_col = get_value_col(df_dev)

print("raw_id_col:", raw_id_col)
print("date_col:", date_col)
print("value_col:", value_col)

if raw_id_col is None:
    print("Nenhuma coluna ID encontrada. Criando ID artificial por linha.")
    df_dev = df_dev.copy()
    df_holdout = df_holdout.copy()
    df_dev["process_node_id"] = [f"dev_proc_{i}" for i in range(len(df_dev))]
    df_holdout["process_node_id"] = [f"holdout_proc_{i}" for i in range(len(df_holdout))]
else:
    df_dev = df_dev.copy()
    df_holdout = df_holdout.copy()
    df_dev["process_node_id"] = "proc_" + df_dev[raw_id_col].astype(str)
    df_holdout["process_node_id"] = "proc_" + df_holdout[raw_id_col].astype(str)

id_col = "process_node_id"
y_dev = df_dev[TARGET_COL].astype(int).values
y_holdout = df_holdout[TARGET_COL].astype(int).values

print("Taxa perda dev:", y_dev.mean())
print("Taxa perda holdout:", y_holdout.mean())

## 1. Seleção das colunas para construir o grafo

O grafo será bipartido:

```text
processo → entidade categórica
```

Exemplos de entidades úteis: assunto, produto, carteira, ação, comarca, UF, fase, órgão julgador, vara, escritório e advogado.

In [ ]:
# ============================================================
# Seleção de colunas categóricas para o grafo
# ============================================================

LEAKAGE_PATTERNS = [
    "target", "desfecho", "resultado", "sentenca", "sentença", "condenacao", "condenação",
    "perdeu", "ganhou", "dataencerramento", "encerramento", "motivo_encerramento",
    "valor_do_acordo", "acordo_valor", "valor_acordo", "valor_condenacao",
    "valor_condenação", "indenizacao", "indenização", "procedente", "improcedente",
]

ID_PATTERNS = ["id", "numero", "número", "cpf", "cnpj", "hash", "process_node_id"]

def is_leakage_col(col):
    c = col.lower()
    return any(p in c for p in LEAKAGE_PATTERNS)

def is_id_like_col(col):
    c = col.lower()
    return any(p in c for p in ID_PATTERNS)

def choose_graph_entity_cols(df, max_cols=30):
    object_cols = df.select_dtypes(include=["object", "category", "string", "bool"]).columns.tolist()
    candidate_cols = []
    for c in object_cols:
        if c in [TARGET_COL, id_col, "process_node_id"]:
            continue
        if is_leakage_col(c) or is_id_like_col(c):
            continue
        nunique = df[c].nunique(dropna=True)
        non_null = df[c].notna().mean()
        if nunique < 2:
            continue
        if nunique > MAX_CATEGORIES_PER_COL:
            continue
        if non_null < 0.02:
            continue
        candidate_cols.append((c, nunique, non_null))
    candidate_cols = sorted(candidate_cols, key=lambda x: (x[1], -x[2]))
    return [c for c, _, _ in candidate_cols[:max_cols]], pd.DataFrame(candidate_cols, columns=["col", "nunique", "non_null_rate"])

df_all_for_graph = pd.concat([df_dev, df_holdout], axis=0, ignore_index=True)
graph_entity_cols, graph_cols_profile = choose_graph_entity_cols(df_all_for_graph)

PREFERRED_GRAPH_COLS = [
    "nm_assunto_norm", "nm_assunto", "assunto", "Nm_Assunto",
    "nm_produto", "produto", "Nm_Produto",
    "carteira", "Carteira",
    "nm_acao", "acao", "ação", "Nm_Acao",
    "comarca", "Comarca",
    "uf", "UF",
    "fase", "fase_processual", "Fasedoprocesso",
    "orgao_julgador", "órgão_julgador", "vara",
    "escritorio", "advogado",
]
manual_cols = [c for c in PREFERRED_GRAPH_COLS if c in df_dev.columns and c in df_holdout.columns and not is_leakage_col(c)]
graph_entity_cols = list(dict.fromkeys(manual_cols + graph_entity_cols))

print("Colunas finais usadas no grafo:", len(graph_entity_cols))
print(graph_entity_cols)
save_report(graph_cols_profile, "90_3f_graph_candidate_columns_profile.csv")
display(graph_cols_profile.head(100))

## 2. Construção do grafo processo–entidade

Arestas criadas:

```text
processo → nome_coluna=valor_categoria
```

Entidades raras são filtradas por `MIN_ENTITY_FREQ` para reduzir ruído e acelerar o embedding.

In [ ]:
# ============================================================
# Construção das arestas processo-entidade
# ============================================================

def build_process_entity_edges(df_all, process_col, entity_cols, min_entity_freq=20):
    rows = []
    for c in entity_cols:
        s = df_all[c].map(normalize_token)
        vc = s.value_counts(dropna=True)
        valid_values = set(vc[vc >= min_entity_freq].index)
        mask = s.isin(valid_values)
        if mask.sum() == 0:
            continue
        tmp = pd.DataFrame({
            "src": df_all.loc[mask, process_col].astype(str).values,
            "dst": [f"{c}={v}" for v in s.loc[mask].values],
            "weight": 1.0,
            "entity_col": c,
        })
        rows.append(tmp)
    if not rows:
        return pd.DataFrame(columns=["src", "dst", "weight", "entity_col"])
    edges = pd.concat(rows, axis=0, ignore_index=True).drop_duplicates(["src", "dst"])
    return edges

t0 = time.time()
edges = build_process_entity_edges(df_all_for_graph, id_col, graph_entity_cols, MIN_ENTITY_FREQ)
print("edges:", edges.shape)
print("tempo:", format_elapsed(time.time() - t0))

edges.to_csv(GRAPH_EDGES_FILE, index=False, encoding="utf-8")
print("Arestas salvas em:", GRAPH_EDGES_FILE)
display(edges.head())

n_processos = df_all_for_graph[id_col].nunique()
n_entity_nodes = edges["dst"].nunique()
n_nodes = pd.concat([edges["src"], edges["dst"]]).nunique()
n_edges = len(edges)

graph_profile = pd.DataFrame([{
    "run_mode": RUN_MODE,
    "n_processos": n_processos,
    "n_entity_nodes": n_entity_nodes,
    "n_nodes_total": n_nodes,
    "n_edges": n_edges,
    "n_entity_cols": len(graph_entity_cols),
    "min_entity_freq": MIN_ENTITY_FREQ,
    "n2v_dim": N2V_DIM,
    "n2v_walk_length": N2V_WALK_LENGTH,
    "n2v_num_walks": N2V_NUM_WALKS,
    "n2v_window": N2V_WINDOW,
    "n2v_epochs": N2V_EPOCHS,
    "has_pecanpy": HAS_PECANPY,
    "has_nodevectors": HAS_NODEVECTORS,
}])
save_report(graph_profile, "91_3f_graph_profile.csv")
graph_profile

## 3. Geração dos embeddings

O notebook tenta Node2Vec acelerado. Se não houver biblioteca disponível, usa SVD para não travar o experimento.

In [ ]:
# ============================================================
# Embeddings com fallback
# ============================================================

def graph_embeddings_svd_fallback(df_all, edges, process_col, dim=32):
    process_nodes = df_all[process_col].astype(str).drop_duplicates().tolist()
    entity_nodes = edges["dst"].astype(str).drop_duplicates().tolist()
    proc_to_idx = {p: i for i, p in enumerate(process_nodes)}
    ent_to_idx = {e: i for i, e in enumerate(entity_nodes)}
    row = edges["src"].map(proc_to_idx)
    col = edges["dst"].map(ent_to_idx)
    mask = row.notna() & col.notna()
    X_pe = sparse.csr_matrix(
        (np.ones(mask.sum(), dtype=np.float32), (row.loc[mask].astype(int).values, col.loc[mask].astype(int).values)),
        shape=(len(process_nodes), len(entity_nodes)),
    )
    n_components = min(dim, max(2, min(X_pe.shape) - 1))
    svd = TruncatedSVD(n_components=n_components, random_state=RANDOM_STATE)
    emb = svd.fit_transform(X_pe)
    emb_cols = [f"graph_emb_{i:03d}" for i in range(emb.shape[1])]
    emb_df = pd.DataFrame(emb, columns=emb_cols)
    emb_df[process_col] = process_nodes
    meta = {
        "method": "svd_fallback",
        "dim_requested": dim,
        "dim_generated": emb.shape[1],
        "explained_variance_sum": float(svd.explained_variance_ratio_.sum()),
    }
    return emb_df, meta

def graph_embeddings_nodevectors(edges, dim=32):
    if not HAS_NETWORKX:
        raise RuntimeError("NetworkX não disponível para nodevectors.")
    if not HAS_NODEVECTORS:
        raise RuntimeError("nodevectors não disponível.")
    G = nx.Graph()
    for r in edges[["src", "dst"]].itertuples(index=False):
        G.add_edge(str(r.src), str(r.dst), weight=1.0)
    g2v = NodevectorsNode2Vec(
        n_components=dim,
        walklen=N2V_WALK_LENGTH,
        epochs=N2V_EPOCHS,
        return_weight=1.0 / max(N2V_P, 1e-12),
        neighbor_weight=1.0 / max(N2V_Q, 1e-12),
        threads=max(os.cpu_count() - 1, 1),
        w2vparams={"window": N2V_WINDOW, "min_count": 1, "workers": max(os.cpu_count() - 1, 1), "sg": 1},
    )
    g2v.fit(G)
    nodes, vectors = [], []
    for node in G.nodes():
        try:
            vectors.append(g2v.predict(node))
            nodes.append(node)
        except Exception:
            pass
    emb = np.vstack(vectors)
    emb_cols = [f"graph_emb_{i:03d}" for i in range(emb.shape[1])]
    emb_df = pd.DataFrame(emb, columns=emb_cols)
    emb_df["node"] = nodes
    meta = {"method": "nodevectors", "dim_requested": dim, "dim_generated": emb.shape[1], "n_nodes_embedded": len(nodes)}
    return emb_df, meta

def graph_embeddings_pecanpy(edges, dim=32):
    if not HAS_PECANPY:
        raise RuntimeError("pecanpy não disponível.")
    from pecanpy import pecanpy as pp
    edge_path = ARTIFACTS_DIR / "tmp_07_03f_pecanpy_edges.edg"
    edges[["src", "dst"]].assign(weight=1.0).to_csv(edge_path, sep="\t", header=False, index=False, encoding="utf-8")
    g = pp.SparseOTF(p=N2V_P, q=N2V_Q, workers=max(os.cpu_count() - 1, 1), verbose=True)
    g.read_edg(str(edge_path), weighted=True, directed=False)
    walks = g.simulate_walks(num_walks=N2V_NUM_WALKS, walk_length=N2V_WALK_LENGTH)
    from gensim.models import Word2Vec
    walks = [[str(x) for x in walk] for walk in walks]
    w2v = Word2Vec(
        sentences=walks,
        vector_size=dim,
        window=N2V_WINDOW,
        min_count=1,
        sg=1,
        workers=max(os.cpu_count() - 1, 1),
        epochs=N2V_EPOCHS,
        seed=RANDOM_STATE,
    )
    nodes = list(w2v.wv.index_to_key)
    emb = np.vstack([w2v.wv[n] for n in nodes])
    emb_cols = [f"graph_emb_{i:03d}" for i in range(emb.shape[1])]
    emb_df = pd.DataFrame(emb, columns=emb_cols)
    emb_df["node"] = nodes
    meta = {"method": "pecanpy", "dim_requested": dim, "dim_generated": emb.shape[1], "n_nodes_embedded": len(nodes)}
    return emb_df, meta

def generate_graph_embeddings(df_all, edges, process_col, dim=32):
    start = time.time()
    if HAS_PECANPY:
        try:
            print("Tentando gerar embeddings com PecanPy...")
            emb_df, meta = graph_embeddings_pecanpy(edges, dim=dim)
            meta["elapsed_sec"] = time.time() - start
            return emb_df, meta
        except Exception as e:
            print("Falha no PecanPy. Erro:", repr(e))
    if HAS_NODEVECTORS:
        try:
            print("Tentando gerar embeddings com nodevectors...")
            emb_df, meta = graph_embeddings_nodevectors(edges, dim=dim)
            meta["elapsed_sec"] = time.time() - start
            return emb_df, meta
        except Exception as e:
            print("Falha no nodevectors. Erro:", repr(e))
    print("Usando fallback com TruncatedSVD.")
    emb_df, meta = graph_embeddings_svd_fallback(df_all, edges, process_col, dim=dim)
    meta["elapsed_sec"] = time.time() - start
    return emb_df, meta

t0 = time.time()
emb_all, emb_meta = generate_graph_embeddings(df_all_for_graph, edges, id_col, dim=N2V_DIM)
print("Método embedding:", emb_meta)
print("Tempo total embedding:", format_elapsed(time.time() - t0))
display(pd.DataFrame([emb_meta]))

In [ ]:
# ============================================================
# Preparar embeddings apenas para processos
# ============================================================

emb_cols = [c for c in emb_all.columns if c.startswith("graph_emb_")]

if "node" in emb_all.columns:
    emb_all_process = emb_all.rename(columns={"node": id_col})
else:
    emb_all_process = emb_all.copy()

process_nodes_all = set(df_all_for_graph[id_col].astype(str).unique())
emb_all_process = emb_all_process[emb_all_process[id_col].astype(str).isin(process_nodes_all)].copy()

df_dev_emb = df_dev[[id_col]].merge(emb_all_process[[id_col] + emb_cols], on=id_col, how="left")
df_holdout_emb = df_holdout[[id_col]].merge(emb_all_process[[id_col] + emb_cols], on=id_col, how="left")

for c in emb_cols:
    df_dev_emb[c] = df_dev_emb[c].fillna(0.0)
    df_holdout_emb[c] = df_holdout_emb[c].fillna(0.0)

save_parquet(df_dev_emb, GRAPH_EMB_DEV_FILE)
save_parquet(df_holdout_emb, GRAPH_EMB_HOLDOUT_FILE)

print("df_dev_emb:", df_dev_emb.shape)
print("df_holdout_emb:", df_holdout_emb.shape)
print("qtd embeddings:", len(emb_cols))

## 4. Montagem da base de modelagem

Este notebook remove features de target encoding por padrão para isolar o sinal do grafo.

In [ ]:
# ============================================================
# Seleção de features tabulares sem target encoding
# ============================================================

def is_target_encoding_feature(col):
    c = col.lower()
    patterns = [
        "target_enc", "target_encoding", "hist_loss_rate", "hist_gain_rate",
        "loss_rate", "gain_rate", "taxa_perda", "taxa_ganho",
        "_rate_smooth", "mestimate", "james_stein",
    ]
    return any(p in c for p in patterns)

def is_forbidden_model_col(col):
    c = col.lower()
    if col in [TARGET_COL, id_col, "process_node_id"]:
        return True
    if is_leakage_col(col):
        return True
    if any(p in c for p in ["texto", "descricao", "descrição", "ementa", "peticao", "petição"]):
        return True
    return False

feature_list_from_file = read_feature_list(FEATURE_LIST_FILE)

if feature_list_from_file:
    base_features = [c for c in feature_list_from_file if c in df_dev.columns and c in df_holdout.columns]
else:
    base_features = [c for c in df_dev.columns if c in df_holdout.columns and not is_forbidden_model_col(c)]

base_features_no_te = [c for c in base_features if not is_forbidden_model_col(c) and not is_target_encoding_feature(c)]

clean_features = []
for c in base_features_no_te:
    if df_dev[c].dtype == "object" or str(df_dev[c].dtype).startswith("string"):
        nunique = df_dev[c].nunique(dropna=True)
        if nunique > MAX_CATEGORIES_PER_COL:
            continue
    clean_features.append(c)

selected_features = clean_features + emb_cols

print("Features tabulares sem TE:", len(clean_features))
print("Embeddings grafo:", len(emb_cols))
print("Features totais:", len(selected_features))

df_dev_model = pd.concat([df_dev.reset_index(drop=True), df_dev_emb[emb_cols].reset_index(drop=True)], axis=1)
df_holdout_model = pd.concat([df_holdout.reset_index(drop=True), df_holdout_emb[emb_cols].reset_index(drop=True)], axis=1)

X_dev = df_dev_model[selected_features].copy()
X_holdout = df_holdout_model[selected_features].copy()

num_cols = [c for c in selected_features if pd.api.types.is_numeric_dtype(X_dev[c])]
cat_cols = [c for c in selected_features if c not in num_cols]

print("num_cols:", len(num_cols))
print("cat_cols:", len(cat_cols))

feature_profile = pd.DataFrame({
    "feature": selected_features,
    "type": ["numeric" if c in num_cols else "categorical" for c in selected_features],
    "is_graph_embedding": [c in emb_cols for c in selected_features],
    "is_target_encoding_like": [is_target_encoding_feature(c) for c in selected_features],
})
save_report(feature_profile, "92_3f_selected_features_profile.csv")
feature_profile.head(100)

## 5. Pipelines e modelos

Modelos testados:

- Logistic Regression;
- Random Forest;
- HistGradientBoosting;
- LightGBM, se disponível;
- XGBoost, se disponível.

In [ ]:
# ============================================================
# Preprocessadores e modelos
# ============================================================

def make_preprocessor(mode):
    if mode == "onehot":
        cat_transformer = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=20 if RUN_MODE == "fast" else None)),
        ])
    elif mode == "ordinal":
        cat_transformer = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("ordinal", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
        ])
    else:
        raise ValueError(f"mode inválido: {mode}")

    num_transformer = Pipeline(steps=[("imputer", SimpleImputer(strategy="median"))])

    return ColumnTransformer(
        transformers=[("num", num_transformer, num_cols), ("cat", cat_transformer, cat_cols)],
        remainder="drop",
        sparse_threshold=0.3,
    )

def make_models():
    models = {}
    models["logistic_onehot_graph"] = {
        "model": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1),
        "preprocess_mode": "onehot",
    }
    models["random_forest_onehot_graph"] = {
        "model": RandomForestClassifier(
            n_estimators=200 if RUN_MODE == "fast" else 500,
            max_depth=10 if RUN_MODE == "fast" else None,
            min_samples_leaf=20,
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        "preprocess_mode": "onehot",
    }
    models["hgb_ordinal_graph"] = {
        "model": HistGradientBoostingClassifier(
            max_iter=150 if RUN_MODE == "fast" else 300,
            learning_rate=0.05,
            max_leaf_nodes=31,
            l2_regularization=0.01,
            random_state=RANDOM_STATE,
            class_weight="balanced",
        ),
        "preprocess_mode": "ordinal",
    }
    if HAS_LIGHTGBM:
        models["lgbm_ordinal_graph"] = {
            "model": lgb.LGBMClassifier(
                n_estimators=300 if RUN_MODE == "fast" else 800,
                learning_rate=0.03,
                num_leaves=31,
                min_child_samples=50,
                subsample=0.8,
                colsample_bytree=0.8,
                class_weight="balanced",
                random_state=RANDOM_STATE,
                n_jobs=-1,
                verbose=-1,
            ),
            "preprocess_mode": "ordinal",
        }
    if HAS_XGBOOST:
        pos = int((y_dev == 1).sum())
        neg = int((y_dev == 0).sum())
        scale_pos_weight = neg / max(pos, 1)
        models["xgb_ordinal_graph"] = {
            "model": xgb.XGBClassifier(
                n_estimators=300 if RUN_MODE == "fast" else 800,
                learning_rate=0.03,
                max_depth=4,
                min_child_weight=10,
                subsample=0.8,
                colsample_bytree=0.8,
                objective="binary:logistic",
                eval_metric="logloss",
                scale_pos_weight=scale_pos_weight,
                random_state=RANDOM_STATE,
                n_jobs=-1,
                tree_method="hist",
            ),
            "preprocess_mode": "ordinal",
        }
    return models

model_specs = make_models()
list(model_specs.keys())

## 6. Walk-forward no Dev

O ranking principal é por `mean_pr_auc_perda`.

In [ ]:
# ============================================================
# Walk-forward no dev
# ============================================================

def sort_by_time_if_possible(X, y, df_ref, date_col):
    if date_col and date_col in df_ref.columns:
        dates = pd.to_datetime(df_ref[date_col], errors="coerce")
        order = np.argsort(dates.fillna(pd.Timestamp("1900-01-01")).values)
        return X.iloc[order].reset_index(drop=True), y[order], df_ref.iloc[order].reset_index(drop=True)
    return X.reset_index(drop=True), y, df_ref.reset_index(drop=True)

X_dev_sorted, y_dev_sorted, df_dev_sorted = sort_by_time_if_possible(X_dev, y_dev, df_dev_model, date_col)
tscv = TimeSeriesSplit(n_splits=N_SPLITS)

results = []
topk_results = []

for model_name, spec in model_specs.items():
    print(f"\nTreinando WF: {model_name}")
    preprocess_mode = spec["preprocess_mode"]
    for fold, (tr_idx, val_idx) in enumerate(tscv.split(X_dev_sorted), start=1):
        X_tr, X_val = X_dev_sorted.iloc[tr_idx], X_dev_sorted.iloc[val_idx]
        y_tr, y_val = y_dev_sorted[tr_idx], y_dev_sorted[val_idx]
        pipe = Pipeline(steps=[("preprocess", make_preprocessor(preprocess_mode)), ("model", clone(spec["model"]))])
        t_fit = time.time()
        pipe.fit(X_tr, y_tr)
        fit_sec = time.time() - t_fit
        proba_val = pipe.predict_proba(X_val)[:, 1]
        metrics = evaluate_probs(y_val, proba_val)
        row = {
            "model": model_name,
            "preprocess_mode": preprocess_mode,
            "fold": fold,
            "fit_sec": fit_sec,
            "taxa_perda_val": y_val.mean(),
            "taxa_ganho_val": 1 - y_val.mean(),
            "qtd_train": len(tr_idx),
            "qtd_val": len(val_idx),
            "qtd_features": len(selected_features),
            "qtd_graph_embeddings": len(emb_cols),
            "graph_embedding_method": emb_meta["method"],
            **metrics,
        }
        results.append(row)
        topk_df = topk_risco_perda_metrics(y_val, proba_val, ks=TOP_KS)
        topk_df["model"] = model_name
        topk_df["preprocess_mode"] = preprocess_mode
        topk_df["fold"] = fold
        topk_results.append(topk_df)
        print(f"fold={fold} | PR-AUC={metrics['pr_auc_perda']:.4f} | ROC-AUC={metrics['roc_auc_perda']:.4f} | fit={format_elapsed(fit_sec)}")

wf_results = pd.DataFrame(results)
wf_topk_results = pd.concat(topk_results, ignore_index=True)

save_report(wf_results, "93_3f_graph_node2vec_wf_results.csv")
save_report(wf_topk_results, "94_3f_graph_node2vec_wf_topk_results.csv")
wf_results.head()

In [ ]:
model_summary = (
    wf_results
    .groupby(["model", "preprocess_mode"], as_index=False)
    .agg(
        mean_roc_auc_perda=("roc_auc_perda", "mean"),
        std_roc_auc_perda=("roc_auc_perda", "std"),
        mean_pr_auc_perda=("pr_auc_perda", "mean"),
        std_pr_auc_perda=("pr_auc_perda", "std"),
        mean_taxa_perda_val=("taxa_perda_val", "mean"),
        mean_best_f1_perda=("best_f1_perda", "mean"),
        mean_best_f1_threshold_perda=("best_f1_threshold_perda", "mean"),
        mean_precision_perda_t05=("precision_perda_t05", "mean"),
        mean_recall_perda_t05=("recall_perda_t05", "mean"),
        mean_f1_perda_t05=("f1_perda_t05", "mean"),
        mean_pred_perda_rate_t05=("pred_perda_rate_t05", "mean"),
        mean_qtd_graph_embeddings=("qtd_graph_embeddings", "mean"),
    )
    .sort_values("mean_pr_auc_perda", ascending=False)
)

save_report(model_summary, "95_3f_graph_node2vec_model_summary.csv")
model_summary

In [ ]:
topk_summary = (
    wf_topk_results
    .groupby(["model", "preprocess_mode", "top_k"], as_index=False)
    .agg(
        mean_precision_perda_at_k=("precision_perda_at_k", "mean"),
        mean_recall_perda_at_k=("recall_perda_at_k", "mean"),
        mean_lift_perda_at_k=("lift_perda_at_k", "mean"),
        mean_taxa_perda_base=("taxa_perda_base", "mean"),
    )
    .sort_values(["top_k", "mean_precision_perda_at_k"], ascending=[True, False])
)

save_report(topk_summary, "96_3f_graph_node2vec_topk_summary.csv")
topk_summary

## 7. Treinar melhor modelo no Dev completo e avaliar Holdout temporal

In [ ]:
best_row = model_summary.iloc[0]
best_model_name = best_row["model"]
best_preprocess_mode = best_row["preprocess_mode"]

print("Melhor modelo por PR-AUC perda no WF:", best_model_name)
print("Preprocessamento:", best_preprocess_mode)
print("Mean PR-AUC perda WF:", best_row["mean_pr_auc_perda"])

best_spec = model_specs[best_model_name]
best_pipeline = Pipeline(steps=[("preprocess", make_preprocessor(best_preprocess_mode)), ("model", clone(best_spec["model"]))])

t_fit = time.time()
best_pipeline.fit(X_dev, y_dev)
fit_sec = time.time() - t_fit

proba_perda_holdout = best_pipeline.predict_proba(X_holdout)[:, 1]
holdout_m = evaluate_probs(y_holdout, proba_perda_holdout)

holdout_metrics = pd.DataFrame([{
    "model": best_model_name,
    "preprocess_mode": best_preprocess_mode,
    "roc_auc_perda": holdout_m["roc_auc_perda"],
    "pr_auc_perda": holdout_m["pr_auc_perda"],
    "taxa_perda_holdout": y_holdout.mean(),
    "taxa_ganho_holdout": 1 - y_holdout.mean(),
    "best_f1_threshold_perda": holdout_m["best_f1_threshold_perda"],
    "best_f1_precision_perda": holdout_m["best_f1_precision_perda"],
    "best_f1_recall_perda": holdout_m["best_f1_recall_perda"],
    "best_f1_perda": holdout_m["best_f1_perda"],
    "precision_perda_t05": holdout_m["precision_perda_t05"],
    "recall_perda_t05": holdout_m["recall_perda_t05"],
    "f1_perda_t05": holdout_m["f1_perda_t05"],
    "pred_perda_rate_t05": holdout_m["pred_perda_rate_t05"],
    "qtd_dev": len(df_dev),
    "qtd_holdout": len(df_holdout),
    "qtd_features": len(selected_features),
    "qtd_graph_embeddings": len(emb_cols),
    "graph_embedding_method": emb_meta["method"],
    "fit_sec": fit_sec,
}])

save_report(holdout_metrics, "97_3f_graph_node2vec_holdout_best_model_metrics.csv")
holdout_metrics

In [ ]:
pred_t05 = (proba_perda_holdout >= 0.5).astype(int)

print("Classification report — threshold 0.5")
print("Classe 0 = banco_ganhou")
print("Classe 1 = banco_perdeu")
print(classification_report(y_holdout, pred_t05, target_names=["banco_ganhou", "banco_perdeu"], digits=4))

print("Confusion matrix — threshold 0.5")
print("Linhas = real | Colunas = previsto")
print(confusion_matrix(y_holdout, pred_t05))

## 8. Holdout top-k — risco puro de perda

In [ ]:
holdout_topk_perda = topk_risco_perda_metrics(y_true=y_holdout, proba_perda=proba_perda_holdout, ks=TOP_KS)
holdout_topk_perda["model"] = best_model_name
holdout_topk_perda["preprocess_mode"] = best_preprocess_mode
holdout_topk_perda["graph_embedding_method"] = emb_meta["method"]

save_report(holdout_topk_perda, "98_3f_graph_node2vec_holdout_topk_risco_perda.csv")
holdout_topk_perda

In [ ]:
ranking_risco = df_holdout_model.copy()
ranking_risco["proba_banco_perdeu"] = proba_perda_holdout
ranking_risco["score_risco_perda"] = proba_perda_holdout
ranking_risco["pred_banco_perdeu_t05"] = pred_t05
ranking_risco = ranking_risco.sort_values("score_risco_perda", ascending=False).reset_index(drop=True)
ranking_risco["rank_risco_perda"] = np.arange(1, len(ranking_risco) + 1)
ranking_risco["percentil_risco_perda"] = ranking_risco["rank_risco_perda"] / len(ranking_risco)

save_parquet(ranking_risco, RANKING_RISCO_HOLDOUT_FILE)
ranking_risco[[id_col, TARGET_COL, "proba_banco_perdeu", "rank_risco_perda", "percentil_risco_perda"]].head(20)

## 9. Holdout top-k — prioridade financeira

Prioridade financeira:

```text
score_financeiro = probabilidade de perda × valor ajuizado
```

In [ ]:
VALUE_COL = get_value_col(df_holdout_model)
print("VALUE_COL:", VALUE_COL)

if VALUE_COL and VALUE_COL in df_holdout_model.columns:
    holdout_topk_financeiro = topk_prioridade_financeira_metrics(
        y_true=y_holdout,
        proba_perda=proba_perda_holdout,
        valor_ajuizado=df_holdout_model[VALUE_COL],
        ks=TOP_KS,
    )
    holdout_topk_financeiro["model"] = best_model_name
    holdout_topk_financeiro["preprocess_mode"] = best_preprocess_mode
    holdout_topk_financeiro["graph_embedding_method"] = emb_meta["method"]
    save_report(holdout_topk_financeiro, "99_3f_graph_node2vec_holdout_topk_prioridade_financeira.csv")
else:
    print("Nenhuma coluna de valor ajuizado encontrada. Pulando análise financeira.")
    holdout_topk_financeiro = pd.DataFrame()

holdout_topk_financeiro

In [ ]:
if VALUE_COL and VALUE_COL in df_holdout_model.columns:
    ranking_financeiro = df_holdout_model.copy()
    ranking_financeiro["proba_banco_perdeu"] = proba_perda_holdout
    ranking_financeiro["valor_ajuizado_modelo"] = safe_numeric_series(ranking_financeiro[VALUE_COL]).fillna(0).clip(lower=0)
    ranking_financeiro["score_prioridade_financeira"] = ranking_financeiro["proba_banco_perdeu"] * ranking_financeiro["valor_ajuizado_modelo"]
    ranking_financeiro = ranking_financeiro.sort_values("score_prioridade_financeira", ascending=False).reset_index(drop=True)
    ranking_financeiro["rank_prioridade_financeira"] = np.arange(1, len(ranking_financeiro) + 1)
    ranking_financeiro["percentil_prioridade_financeira"] = ranking_financeiro["rank_prioridade_financeira"] / len(ranking_financeiro)
    save_parquet(ranking_financeiro, RANKING_FIN_HOLDOUT_FILE)
    display(ranking_financeiro[[id_col, TARGET_COL, "proba_banco_perdeu", "valor_ajuizado_modelo", "score_prioridade_financeira", "rank_prioridade_financeira"]].head(20))
else:
    ranking_financeiro = pd.DataFrame()

## 10. Comparação com etapas anteriores

Preencha manualmente os resultados anteriores quando quiser comparar contra 3A, 3B, 3C e 3D.

In [ ]:
previous_steps = [
    {"notebook": "3A_baseline", "model": "random_forest_onehot_tfidf", "holdout_pr_auc_perda": np.nan, "holdout_roc_auc_perda": np.nan, "top5_precision_perda": np.nan, "top10_fin_share_valor_perdas": np.nan},
    {"notebook": "3B_encoders", "model": "hgb_mestimate_encoder", "holdout_pr_auc_perda": np.nan, "holdout_roc_auc_perda": np.nan, "top5_precision_perda": np.nan, "top10_fin_share_valor_perdas": np.nan},
    {"notebook": "3C_lgbm_xgb", "model": "xgb_ordinal", "holdout_pr_auc_perda": np.nan, "holdout_roc_auc_perda": np.nan, "top5_precision_perda": np.nan, "top10_fin_share_valor_perdas": np.nan},
    {"notebook": "3D_text", "model": "xgb_ordinal_tfidf", "holdout_pr_auc_perda": np.nan, "holdout_roc_auc_perda": np.nan, "top5_precision_perda": np.nan, "top10_fin_share_valor_perdas": np.nan},
]

top5 = holdout_topk_perda.loc[holdout_topk_perda["top_k"] == 0.05, "precision_perda_at_k"]
top5_val = float(top5.iloc[0]) if len(top5) else np.nan

if not holdout_topk_financeiro.empty:
    top10fin = holdout_topk_financeiro.loc[holdout_topk_financeiro["top_k"] == 0.10, "share_valor_perdas_total"]
    top10fin_val = float(top10fin.iloc[0]) if len(top10fin) else np.nan
else:
    top10fin_val = np.nan

current_row = {
    "notebook": "3F_graph_node2vec_fast",
    "model": best_model_name,
    "holdout_pr_auc_perda": holdout_m["pr_auc_perda"],
    "holdout_roc_auc_perda": holdout_m["roc_auc_perda"],
    "top5_precision_perda": top5_val,
    "top10_fin_share_valor_perdas": top10fin_val,
}

comparison = pd.DataFrame(previous_steps + [current_row])
base_idx = comparison["notebook"].eq("3A_baseline")
if base_idx.any():
    base = comparison.loc[base_idx].iloc[0]
    for metric in ["holdout_pr_auc_perda", "holdout_roc_auc_perda", "top5_precision_perda", "top10_fin_share_valor_perdas"]:
        comparison[f"delta_{metric}_vs_3a"] = comparison[metric] - base[metric]

save_report(comparison, "100_3f_graph_node2vec_comparison_previous_steps.csv")
comparison

## 11. Summary final

Use este resumo para decidir se vale criar o próximo notebook híbrido com target encoding + texto + graph embeddings.

In [ ]:
summary_step_3f = pd.DataFrame([{
    "target_semantica": "0=banco_ganhou | 1=banco_perdeu",
    "notebook": "07_03F_graph_node2vec_fast_features_early_stage",
    "run_mode": RUN_MODE,
    "graph_embedding_method": emb_meta["method"],
    "has_pecanpy": HAS_PECANPY,
    "has_nodevectors": HAS_NODEVECTORS,
    "n2v_dim": N2V_DIM,
    "n2v_walk_length": N2V_WALK_LENGTH,
    "n2v_num_walks": N2V_NUM_WALKS,
    "n2v_window": N2V_WINDOW,
    "n2v_epochs": N2V_EPOCHS,
    "min_entity_freq": MIN_ENTITY_FREQ,
    "qtd_graph_entity_cols": len(graph_entity_cols),
    "qtd_graph_edges": len(edges),
    "qtd_graph_nodes": n_nodes,
    "qtd_graph_embeddings": len(emb_cols),
    "best_model_3f_walk_forward": best_model_name,
    "best_model_3f_preprocess": best_preprocess_mode,
    "best_model_3f_mean_pr_auc_perda_wf": float(best_row["mean_pr_auc_perda"]),
    "best_model_3f_mean_roc_auc_perda_wf": float(best_row["mean_roc_auc_perda"]),
    "holdout_pr_auc_perda": holdout_m["pr_auc_perda"],
    "holdout_roc_auc_perda": holdout_m["roc_auc_perda"],
    "taxa_perda_holdout": y_holdout.mean(),
    "taxa_ganho_holdout": 1 - y_holdout.mean(),
    "qtd_features": len(selected_features),
    "qtd_dev": len(df_dev),
    "qtd_holdout": len(df_holdout),
    "ranking_risco_holdout_path": str(RANKING_RISCO_HOLDOUT_FILE),
    "ranking_financeiro_holdout_path": str(RANKING_FIN_HOLDOUT_FILE) if VALUE_COL else None,
}])

save_report(summary_step_3f, "101_summary_step_3f_graph_node2vec_fast.csv")
summary_step_3f.T

In [ ]:
model_path = MODELS_DIR / "best_model_07_03f_graph_node2vec_fast.pkl"
with open(model_path, "wb") as f:
    pickle.dump({
        "pipeline": best_pipeline,
        "selected_features": selected_features,
        "graph_embedding_cols": emb_cols,
        "graph_entity_cols": graph_entity_cols,
        "embedding_meta": emb_meta,
        "best_model_name": best_model_name,
        "best_preprocess_mode": best_preprocess_mode,
        "target_col": TARGET_COL,
        "id_col": id_col,
        "raw_id_col": raw_id_col,
        "value_col": VALUE_COL,
    }, f)

print("Modelo salvo em:", model_path)

## Decisão após rodar

Critério prático:

- Se `graph_embedding_method = pecanpy` ou `nodevectors` e o resultado melhorar top-k ou PR-AUC, vale avançar para o híbrido.
- Se cair em `svd_fallback`, compare contra o notebook SVD anterior, mas ainda não conclua sobre Node2Vec.
- Se o grafo melhorar principalmente `precision@top 1%`, `precision@top 5%` ou `share_valor_perdas_top10`, pode justificar uso mesmo sem grande ganho global de PR-AUC.